# Notebook 36 — True Universality Test

Goal:
- Test whether 2D scaling can collapse back to **true 1D**
- Try alternative effective coordinates

Test forms:
- gamma_eff' = gamma_eff * (T/Tc)^nu
- gamma_eff' = gamma_eff + c/T

Compare collapse quality (MSE, visual alignment)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize

# assume these are already computed from previous notebooks
# gamma_eff_grid, S_grid, T_values

# Placeholder synthetic example (replace with real data)
gamma_eff = np.linspace(0.001, 0.2, 50)
T_values = np.array([18, 22, 26, 30, 34])

def generate_fake_S(g, T):
    return np.exp(-g * (T/26)) / (1 + 0.1/T)

data = []
for T in T_values:
    S = generate_fake_S(gamma_eff, T)
    data.append((gamma_eff.copy(), S.copy(), T))

## Model 1: Power-law rescaling

In [ ]:
def collapse_error_power(nu):
    xs, ys = [], []
    for g, S, T in data:
        x = g * (T/26)**nu
        xs.append(x)
        ys.append(S)
    xs = np.concatenate(xs)
    ys = np.concatenate(ys)
    
    # simple binning collapse metric
    bins = np.linspace(xs.min(), xs.max(), 50)
    digitized = np.digitize(xs, bins)
    means = [ys[digitized == i].mean() if np.any(digitized==i) else 0 for i in range(len(bins))]
    residual = 0
    for g, S, T in data:
        x = g * (T/26)**nu
        idx = np.digitize(x, bins)
        pred = np.array([means[i-1] if i>0 and i<len(means) else 0 for i in idx])
        residual += np.mean((S - pred)**2)
    return residual

res = minimize(collapse_error_power, x0=[1.0])
nu_opt = res.x[0]
print('Optimal nu:', nu_opt)

## Model 2: Additive correction

In [ ]:
def collapse_error_additive(c):
    xs, ys = [], []
    for g, S, T in data:
        x = g + c/T
        xs.append(x)
        ys.append(S)
    xs = np.concatenate(xs)
    ys = np.concatenate(ys)
    
    bins = np.linspace(xs.min(), xs.max(), 50)
    digitized = np.digitize(xs, bins)
    means = [ys[digitized == i].mean() if np.any(digitized==i) else 0 for i in range(len(bins))]
    residual = 0
    for g, S, T in data:
        x = g + c/T
        idx = np.digitize(x, bins)
        pred = np.array([means[i-1] if i>0 and i<len(means) else 0 for i in idx])
        residual += np.mean((S - pred)**2)
    return residual

res2 = minimize(collapse_error_additive, x0=[0.1])
c_opt = res2.x[0]
print('Optimal c:', c_opt)

## Visualization

In [ ]:
plt.figure()
for g, S, T in data:
    x = g * (T/26)**nu_opt
    plt.plot(x, S, label=f'T={T}')

plt.xlabel('collapsed coordinate (power-law)')
plt.ylabel('S')
plt.title('Attempted true 1D collapse (power-law)')
plt.legend()
plt.show()

plt.figure()
for g, S, T in data:
    x = g + c_opt/T
    plt.plot(x, S, label=f'T={T}')

plt.xlabel('collapsed coordinate (additive)')
plt.ylabel('S')
plt.title('Attempted true 1D collapse (additive)')
plt.legend()
plt.show()